Data Pipelines with Lakeflow Spark Declarative Pipelines (SDP)
This course introduces users to the essential concepts and skills needed to build data pipelines using Lakeflow Spark Declarative Pipelines (SDP) in Databricks for incremental batch or streaming ingestion and processing through multiple streaming tables and materialized views. Designed for data engineers new to Spark Declarative Pipelines, the course provides a comprehensive overview of core components such as incremental data processing, streaming tables, materialized views, and temporary views, highlighting their specific purposes and differences.

Topics covered include:

Developing and debugging ETL pipelines with the multi-file editor in Spark Declarative Pipelines using SQL (with Python code examples provided)

How Spark Declarative Pipelines track data dependencies in a pipeline through the pipeline graph

Configuring pipeline compute resources, data assets, trigger modes, and other advanced options

Next, the course introduces data quality expectations in Spark Declarative Pipelines, guiding users through the process of integrating expectations into pipelines to validate and enforce data integrity. Learners will then explore how to put a pipeline into production, including scheduling options, and enabling pipeline event logging to monitor pipeline performance and health.

Finally, the course covers how to implement Change Data Capture (CDC) using the AUTO CDC INTO syntax within Spark Declarative Pipelines to manage slowly changing dimensions (SCD Type 1 and Type 2), preparing users to integrate CDC into their own pipelines.

Course Agenda
The following modules are part of the Data Engineer Learning Path by Databricks Academy.

#	Notebook Name
1	REQUIRED - Course Setup and Creating a Pipeline
2	Developing a Simple Pipeline
3	Adding Data Quality Expectations
4L	Create a Pipeline
5	Deploying a Pipeline to Production
6	Change Data Capture with AUTO CDC with SCD TYPE 1
7L	AUTO CDC INTO with SCD Type 1
Requirements
Please review the following requirements before starting the lesson:

To run demo and lab notebooks, you need to use the following Databricks runtime: 17.3.x-scala2.13



1 - REQUIRED - Course Setup and Creating a Pipeline
In this demo, we'll set up the course environment, explore its components, build a traditional ETL pipeline using JSON files as the data source, and then learn how to create a sample Lakeflow Spark Declarative Pipeline (SDP).

Learning Objectives
By the end of this lesson, you will be able to:

Efficiently navigate the Workspace to locate course catalogs, schemas, and source files.
Create a Lakeflow Spark Declarative Pipeline using the Workspace and the Pipeline UI.
IMPORTANT - PLEASE READ!
REQUIRED - This notebook is required for all users to run. If you do not run this notebook, you will be missing the necessary files and schemas required for the rest of the course.

REQUIRED - SELECT CLASSIC COMPUTE (your cluster starts with labuser)
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that Serverless is enabled by default.

Follow these steps to select the classic compute cluster:

Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use Serverless.

If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

In the drop-down, select More.

In the Attach to an existing compute resource pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

NOTE: If your cluster has terminated, you might need to restart it in order to select it. To do this:

Right-click on Compute in the left navigation pane and select Open in new tab.

Find the triangle icon to the right of your compute cluster name and click it.

Wait a few minutes for the cluster to start.

Once the cluster is running, complete the steps above to select your cluster.

A. Classroom Setup
Run the following cell to configure your working environment for this course.

NOTE: The DA object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically create and reference the information needed to run the course.



In [0]:
%sql
./Includes/Classroom-Setup-1-setup-REQUIRED

B. Explore the Lab Environment
Explore the raw data source files, catalogs, and schema in the course lab environment.

Complete these steps to explore your user catalog and schemas you will be using in this course:

a. Select the Catalog icon Catalog Icon in the left navigation bar.

b. You should see your unique catalog, named something like labuser1234_56789. You will use this catalog throughout the course.

c. Expand your labuser catalog. It should contain the following schemas:

1_bronze_db
2_silver_db
3_gold_db
default

Complete the following steps to view where our streaming raw source files are coming from:

a. Select the Catalog icon Catalog Icon in the left navigation bar.

b. Expand the dbacademy catalog.

c. Expand the ops schema and then Volumes.

d. Expand your labuser@vocareum volume. You should notice that your volume contains three folders:

customers
orders
status
e. Expand each folder and notice that each cloud storage location contains a single JSON file to start with.

3. To easily reference this volume path (/Volumes/dbacademy/ops/your-labuser-name) throughout the course, you can use the:

The python DA.paths.working_dir variable
The SQL DA.paths_working_dir variable
Run the cells below and confirm that the path points to your volume.

Example: /Volumes/dbacademy/ops/labuser1234_5678@vocareum

In [0]:
## With Python
print(DA.paths.working_dir)

## RESULT :-  /Volumes/dbacademy/ops/labuser13765677_1770877095@vocareum_com

In [0]:
%sql
-- With SQL
values(DA.paths_working_dir)

---RESULT : /Volumes/dbacademy/ops/labuser13765677_1770877095@vocareum_com

Query the raw JSON file(s) in your /Volumes/dbacademy/ops/your-labuser-name/orders volume to preview the data.

Notice that the JSON file is displayed ingested into tabular form using the read_files function. Take note of the following:

a. The orders JSON file contains order data for a company.

b. The one JSON file in your /orders volume (00.json) contains 174 rows. Remember that number for later.


In [0]:
spark.sql(f'''
          SELECT * 
          FROM json.`{DA.paths.working_dir}/orders`
          ''').display()

RSESULT:
customer_id	notifications	order_id	order_timestamp
23094	Y	75123	1640392092
23457	N	75124	1640392500
23564	Y	75125	1640394862
23392	N	75126	1640396067
23101	Y	75127	1640399066
23466	N	75128	1640404853
23834	Y	75129	1640407272
23852	Y	75130	1640419989
23483	Y	75131	1640422131
23821	N	75132	1640423697
23907	Y	75133	1640427687
23088	Y	75134	1640427813
23327	Y	75135	1640432499
23154	Y	75136	1640433629
23818	Y	75137	1640434907
23568	Y	75138	1640438599
23599	Y	75139	1640441108
23646	Y	75140	1640442728
23447	Y	75141	1640442982
22196	N	75142	1640444289
23004	Y	75143	1640444735
22457	Y	75144	1640453556
23560	N	75145	1640454027
23892	Y	75146	1640469617
23270	N	75147	1640469849
23318	Y	75148	1640470511
23134	Y	75149	1640476865
23631	Y	75150	1640479937
23447	Y	75151	1640480642
22980	Y	75152	1640489317
23662	Y	75153	1640490661
23128	Y	75154	1640491953
23305	Y	75155	1640500820
23790	Y	75156	1640506768
23673	Y	75157	1640514816
22447	Y	75158	1640516867
22600	Y	75159	1640516913
23598	Y	75160	1640517492
23453	N	75161	1640521294
22477	N	75162	1640526465
23863	Y	75163	1640526485
22485	N	75164	1640526860
23786	N	75165	1640532784
23833	N	75166	1640534000
23242	N	75167	1640534721
23120	Y	75168	1640548053
23815	Y	75169	1640548610
22757	Y	75170	1640548846
23748	N	75171	1640552043
23416	Y	75172	1640552112
23218	N	75173	1640553981
23867	Y	75174	1640564080
23259	N	75175	1640567847
22757	N	75176	1640569060
22599	Y	75177	1640574553
23606	Y	75178	1640575274
23493	N	75179	1640575962
22467	Y	75180	1640580049
23828	Y	75181	1640582140
22316	Y	75182	1640587536
23061	Y	75183	1640594017
23813	Y	75184	1640603248
23358	Y	75185	1640611591
23778	Y	75186	1640613010
23477	Y	75187	1640618976
23179	Y	75188	1640620853
23769	Y	75189	1640625214
23836	N	75190	1640625322
22557	Y	75191	1640625597
23359	Y	75192	1640631831
23768	Y	75193	1640631849
23669	Y	75194	1640633377
23823	Y	75195	1640637588
22296	Y	75196	1640642842
23318	Y	75197	1640643224
23219	N	75198	1640643721
23072	Y	75199	1640653481
23910	Y	75200	1640660661
23833	Y	75201	1640665029
23696	Y	75202	1640665868
23815	Y	75203	1640665971
23195	N	75204	1640671232
23183	Y	75205	1640672063
23367	N	75206	1640675667
23088	Y	75207	1640675687
22752	Y	75208	1640678086
23326	Y	75209	1640681064
23212	Y	75210	1640682880
23524	Y	75211	1640684944
23151	Y	75212	1640690706
23582	Y	75213	1640691368
23182	Y	75214	1640691634
22826	Y	75215	1640696465
23158	Y	75216	1640706768
23343	Y	75217	1640717605
23390	Y	75218	1640721908
23574	Y	75219	1640725566
23282	Y	75220	1640726897
23206	N	75221	1640730881
23811	Y	75222	1640731443
23734	Y	75223	1640732281
23284	N	75224	1640737911
23397	Y	75225	1640741859
23285	Y	75226	1640743461
23443	Y	75227	1640744138
23841	Y	75228	1640745536
23489	Y	75229	1640745939
23377	Y	75230	1640751789
23519	Y	75231	1640753686
23273	Y	75232	1640754906
23942	Y	75233	1640755409
23714	Y	75234	1640755921
22959	N	75235	1640758650
23642	Y	75236	1640763112
23277	N	75237	1640781747
23777	Y	75238	1640790253
23748	N	75239	1640799146
23511	Y	75240	1640799965
23964	N	75241	1640800237
23542	Y	75242	1640803027
23842	Y	75243	1640805315
23371	Y	75244	1640808136
22895	Y	75245	1640809481
23885	Y	75246	1640810227
23875	Y	75247	1640812303
23331	Y	75248	1640815265
23218	N	75249	1640828420
23779	N	75250	1640830827
23187	N	75251	1640837025
23435	Y	75252	1640837290
23124	Y	75253	1640837302
23215	N	75254	1640852084
23863	Y	75255	1640857299
23800	Y	75256	1640858077
23503	N	75257	1640859758
23755	Y	75258	1640862499
23509	Y	75259	1640864646
22338	N	75260	1640867193
22935	Y	75261	1640869044
23311	Y	75262	1640869535
23819	Y	75263	1640869633
23312	Y	75264	1640886912
23484	Y	75265	1640888803
23290	Y	75266	1640889524
23558	Y	75267	1640889752
23086	Y	75268	1640893693
23822	Y	75269	1640895938
23212	Y	75270	1640898363
23715	Y	75271	1640902531
22161	Y	75272	1640904968
23882	Y	75273	1640909763
23147	Y	75274	1640911244
23577	Y	75275	1640911324
23888	Y	75276	1640914350
22815	Y	75277	1640922137
23521	Y	75278	1640933033
23887	Y	75279	1640933550
23551	N	75280	1640933653
23886	Y	75281	1640935059
23376	Y	75282	1640935106
23911	N	75283	1640941395
23310	Y	75284	1640948702
23109	N	75285	1640950790
23291	Y	75286	1640953620
23556	Y	75287	1640953917
23581	Y	75288	1640958185
23621	Y	75289	1640960321
23365	Y	75290	1640963057
23716	N	75291	1640971896
23206	Y	75292	1640974381
23027	Y	75293	1640984029
23380	N	75294	1640987218
22701	Y	75295	1640987527
23766	Y	75296	1640990674

Traditionally, you would build an ETL pipeline by reading all of the files within the cloud storage location each time the pipeline runs. As data scales, this method becomes inefficient, more expensive, and time-consuming.

For example, you would write code like below.

NOTES:

The tables and views will be written to your labuser.default schema (database).
Knowledge of the Databricks read_files function is prerequisite for this course.

In [0]:
%sql
-- JSON -> Bronze
-- Read ALL files from your working directory each time the query is executed
CREATE OR REPLACE TABLE default.orders_bronze
AS 
SELECT 
  *,
  current_timestamp() AS processing_time,
  _metadata.file_name AS source_file
FROM read_files(
    DA.paths_working_dir || "/orders", 
    format =>"json");


-- Bronze -> Silver
-- Read the entire bronze table each time the query is executed
CREATE OR REPLACE TABLE default.orders_silver
AS 
SELECT 
  order_id,
  timestamp(order_timestamp) AS order_timestamp, 
  customer_id,
  notifications
FROM default.orders_bronze;   


-- Silver -> Gold
-- Aggregate the silver each time the query is executed.
CREATE OR REPLACE VIEW default.orders_by_date_vw     
AS 
SELECT 
  date(order_timestamp) AS order_date, 
  count(*) AS total_daily_orders
FROM default.orders_silver                               
GROUP BY date(order_timestamp);

3. Run the code in the cells to view the orders_bronze and orders_silver tables, and the orders_by_date_vw view. Explore the results.

In [0]:
%sql
SELECT *
FROM default.orders_bronze
LIMIT 5;